In [1]:
import pandas as pd
import numpy as np

In [2]:
file_path =  r"D:\Managed Workspace\Project\Customer Experience Survey Analytics\Data Set\survey_data_raw.csv"

In [3]:
df = pd.read_csv(file_path)

In [4]:
df.head()

,respondent_id,full_name,email,age,gender,city,state,survey_date,product_purchased,purchase_channel,satisfaction_score,nps_score,delivery_rating,support_rating,monthly_spend_inr,income_band,competitors_used,referred_by,would_recommend,feedback_comment
0,R0339,Amit Mehta,amit.mehta94@gmail.com,59,Other,Chennai,Tamil Nadu,2026-03-20,Smartphone,Store,4,5.0,5.0,5.0,42759,10-20L,"Tata Cliq, Meesho",R0135,No,Average experience
1,R0206,Rahul Iyer,rahul.iyer49@outlook.com,30,Other,Lucknow,Uttar Pradesh,04-04-26,Speaker,Store,2,3.0,1.0,4.0,13493,6-10L,Amazon,R0006,No,Average experience
2,R0278,Pooja Singh,pooja.singh10@yahoo.com,43,Male,Bangalore,Karnataka,05/02/2026,speaker,Store,NaN,6.0,1.0,5.0,7035,10-20L,"Amazon, Meesho, Tata Cliq",R0207,Yes,Support team was unhelpful
3,R0348,Tanvi Khan,tanvi.khan50@gmail.com,-5,MALE,Pune,Maharashtra,2026-05-05,Headphones,App,4,8.0,4.0,5.0,26311,20L+,NaN,R0069,No,Excellent service
4,R0419,Suresh Sharma,suresh.sharma32outlook.com,23,Other,Calcutta,West Bengal,2026-02-17,Smartphone,Marketplace,4,9.0,4.0,3.0,16318,20L+,"Amazon, Croma, Tata Cliq",NaN,No,Value for money


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 20 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   respondent_id       500 non-null    object 
 1   full_name           500 non-null    object 
 2   email               469 non-null    object 
 3   age                 482 non-null    object 
 4   gender              500 non-null    object 
 5   city                500 non-null    object 
 6   state               500 non-null    object 
 7   survey_date         500 non-null    object 
 8   product_purchased   500 non-null    object 
 9   purchase_channel    500 non-null    object 
 10  satisfaction_score  495 non-null    object 
 11  nps_score           492 non-null    float64
 12  delivery_rating     481 non-null    float64
 13  support_rating      474 non-null    float64
 14  monthly_spend_inr   493 non-null    object 
 15  income_band         500 non-null    object 
 16  competit

In [6]:
print("Duplicate IDs count:", df['respondent_id'].duplicated().sum())

Duplicate IDs count: 10


In [7]:
duplicate_rows = df[df.duplicated(subset=['respondent_id'], keep=False)].sort_values('respondent_id')

In [8]:
df = df.drop_duplicates(subset=['respondent_id'], keep='first')

In [9]:
df = df.drop(columns=['full_name', 'email'])

In [10]:
print("New duplicate count:", df['respondent_id'].duplicated().sum())
print("Total rows left in dataset:", len(df))

New duplicate count: 0
Total rows left in dataset: 490


In [11]:
df['age'] = df['age'].replace('thirty', 30)

In [12]:
df["age"] = df["age"].replace([-5, 150, 999], 50)

In [13]:
df["age"] = pd.to_numeric(df["age"], errors="coerce")

In [14]:
df["age"] = df["age"].replace([-5, 150, 999], 50)

In [15]:
age_bins = [17, 24, 34, 54, float("inf")]
age_labels = ["18-24", "25-34", "35-54", "55+"]

In [16]:
df["Age_Group"] = pd.cut(df["age"], bins=age_bins, labels=age_labels)
df["Age_Group"] = df["Age_Group"].fillna("25-34")

In [17]:
gender_map = {
    'Male': 'Male', 'MALE': 'Male', 'M': 'Male', 'male': 'Male',
    'Female': 'Female', 'female': 'Female', 'F': 'Female',
    'Other': 'Other'
}

df['gender'] = df['gender'].map(gender_map).fillna('Other')
print(df['gender'].unique())

['Other' 'Male' 'Female']


In [18]:
df['city'] = df['city'].str.strip().str.title()
df['city'] = df['city'].replace({
    'Bangalore': 'Bengaluru', 'Blr': 'Bengaluru',
    'Calcutta': 'Kolkata',
    'Bombay': 'Mumbai', 'Mumbay': 'Mumbai',
    'Madras': 'Chennai',
    'Poona': 'Pune',
    'Hydrabad': 'Hyderabad',
    'Lakhnau': 'Lucknow',
    'New Delhi': 'Delhi'
})

print(df['city'].unique())

['Chennai' 'Lucknow' 'Bengaluru' 'Pune' 'Kolkata' 'Jaipur' 'Saharanpur'
 'Hyderabad' 'Mumbai' 'Delhi']


In [19]:
def parse_date(x):
    x = str(x).strip()
    for fmt in ('%Y-%m-%d', '%d/%m/%Y', '%d-%b-%Y', '%m-%d-%y'):
        try:
            return pd.to_datetime(x, format=fmt)
        except ValueError:
            continue
    return pd.NaT

df['survey_date'] = df['survey_date'].apply(parse_date)
df['survey_month'] = df['survey_date'].dt.to_period('M').astype(str) 
print(df['survey_date'].isna().sum()) 

0


In [20]:
df['product_purchased'] = df['product_purchased'].str.strip().str.title()
df['product_purchased'].unique()

array(['Smartphone', 'Speaker', 'Headphones', 'Tablet', 'Smart Watch',
       'Laptop'], dtype=object)

In [21]:
df['purchase_channel'].unique()

array(['Store', 'App', 'Marketplace', 'Online'], dtype=object)

In [22]:
df['satisfaction_score'] = df['satisfaction_score'].replace({'Five': 5})
df['satisfaction_score'] = pd.to_numeric(df['satisfaction_score'], errors='coerce')
df['satisfaction_score'] = df['satisfaction_score'].replace({6: None})
df['satisfaction_score'].unique()

array([4.0, 2.0, nan, 1.0, 3.0, 5.0, 4.5, None, 0.0], dtype=object)

In [23]:
df['nps_score'] = pd.to_numeric(df['nps_score'], errors='coerce')
df['nps_score'] = df['nps_score'].where(df['nps_score'].between(0, 10)).fillna(0)
bins = [-1, 6, 8, 10]
labels = ['Detractor', 'Passive', 'Promoter']
df['nps_category'] = pd.cut(df['nps_score'], bins=bins, labels=labels)

In [24]:
df['delivery_rating'] = pd.to_numeric(df['delivery_rating'], errors='coerce')
df['delivery_missing'] = df['delivery_rating'].isna()
df['delivery_rating'].unique()

array([ 5.,  1.,  4.,  3.,  2., nan])

In [25]:
df['support_rating'] = pd.to_numeric(df['support_rating'], errors='coerce')
df['support_rating'].unique()

array([ 5.,  4.,  3.,  1., nan,  2.])

In [26]:
s = (df['monthly_spend_inr'].astype(str)
       .str.replace(r'[₹,]|/-', '', regex=True))
df['monthly_spend_inr'] = pd.to_numeric(s, errors='coerce')

# invalid: negative
df.loc[df['monthly_spend_inr'] < 0, 'monthly_spend_inr'] = np.nan

In [27]:
df['income_band'].unique()

array(['10-20L', '6-10L', '20L+', '<3L', '3-6L'], dtype=object)

In [28]:
df['competitors_used'] = df['competitors_used'].fillna('Unknown')
app_counts = df['competitors_used'].str.get_dummies(sep=', ')
print(app_counts.sum().sort_values(ascending=False))

Unknown             138
Croma               136
Flipkart            119
Tata Cliq           119
Meesho              118
Reliance Digital    115
Amazon              113
dtype: int64


In [29]:
df['referred_by'] = df['referred_by'].fillna('Unknown')
df['referred_by'].unique()

array(['R0135', 'R0006', 'R0207', 'R0069', 'Unknown', 'R0052', 'R0083',
       'R0012', 'R0223', 'R0103', 'R0196', 'R0016', 'R0033', 'R0065',
       'R0057', 'R0034', 'R0040', 'R0076', 'R0097', 'R0025', 'R0038',
       'R0089', 'R0007', 'R0109', 'R0175', 'R0018', 'R0144', 'R0098',
       'R0093', 'R0086', 'R0133', 'R0037', 'R0198', 'R0005', 'R0299',
       'R0244', 'R0150', 'R0116', 'R0398', 'R0031', 'R0091', 'R0383',
       'R0263', 'R0056', 'R0421', 'R0021', 'R0213', 'R0241', 'R0015',
       'R0237', 'R0185', 'R0119', 'R0240', 'R0004', 'R0170', 'R0110',
       'R0115', 'R0001', 'R0171', 'R0386', 'R0346', 'R0036', 'R0088',
       'R0068', 'R0087', 'R0289', 'R0139', 'R0210', 'R0293', 'R0126',
       'R0162', 'R0423', 'R0243', 'R0168', 'R0221', 'R0010', 'R0019',
       'R0028', 'R0101', 'R0279', 'R0046', 'R0047', 'R0071', 'R0351',
       'R0125', 'R0257', 'R0190', 'R0143', 'R0247', 'R0045', 'R0322',
       'R0102', 'R0080', 'R0075', 'R0380', 'R0032', 'R0072', 'R0024',
       'R0008', 'R

In [30]:
df['would_recommend'] = df['would_recommend'].astype(str).str.lower().str.strip()
df['would_recommend'] = df['would_recommend'].replace({
    'y': 'Yes', 
    'yes': 'Yes',
    'n': 'No', 
    'no': 'No',
    'maybe': 'Maybe',
    'nan': 'Maybe' 
})

df['would_recommend'].unique()

array(['No', 'Yes', 'Maybe'], dtype=object)

In [31]:
final_df = pd.concat([df, app_counts], axis=1)
final_df.to_csv('Clean_survey_data.csv', index=False)